### 1. Imports and Configuration

In [ ]:
import gc
import json
from typing import Dict, List, Optional, Tuple
import warnings

import matplotlib.pyplot as plt
import numpy               as np
import pandas              as pd
import torch
import torch.nn            as nn
import torch.nn.functional as F
import torch.optim         as optim
from tqdm import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device : {device}')


def clean_memory():
    '''Clean GPU cache and run garbage collection.'''
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

### 2. Tversky Projection Layer

In [ ]:
class TverskyProjectionLayer(nn.Module):
    '''
    Tversky Projection Layer.

    Computes similarity between input vectors and learned prototypes using
    the differentiable Tversky contrast model:

        S(A, B) = theta * f(A intersect B)
                - alpha * f(A - B)
                - beta  * f(B - A)

    Parameters
    ----------
    in_features            : input dimension
    out_features           : number of prototypes (output dimension)
    num_features           : size of the learnable feature bank
    intersection_reduction : how to aggregate common features
                             ['product', 'min', 'max', 'mean']
    difference_reduction   : how to measure distinctive features
                             ['ignorematch', 'subtractmatch']
    '''

    def __init__(
        self,
        in_features: int,
        out_features: int,
        num_features: int,
        intersection_reduction: str = 'product',
        difference_reduction: str = 'ignorematch',
    ):
        super().__init__()

        valid_intersections = ['product', 'min', 'max', 'mean']
        valid_differences   = ['ignorematch', 'subtractmatch']
        if intersection_reduction not in valid_intersections:
            raise ValueError(f'intersection_reduction must be one of {valid_intersections}')
        if difference_reduction not in valid_differences:
            raise ValueError(f'difference_reduction must be one of {valid_differences}')

        self.in_features             = in_features
        self.out_features            = out_features
        self.num_features            = num_features
        self.intersection_reduction  = intersection_reduction
        self.difference_reduction    = difference_reduction

        # -----------------------------------------------------------------------
        # Learnable parameters. theta / alpha / beta are kept positive
        # via .abs() in forward so the Tversky formula retains its sign
        # semantics throughout training.
        # -----------------------------------------------------------------------

        self.prototypes = nn.Parameter(torch.empty(out_features, in_features))
        self.features   = nn.Parameter(torch.empty(num_features, in_features))
        self.theta      = nn.Parameter(torch.tensor(1.0))
        self.alpha      = nn.Parameter(torch.tensor(0.5))
        self.beta       = nn.Parameter(torch.tensor(0.5))

        nn.init.xavier_uniform_(self.prototypes)
        nn.init.xavier_uniform_(self.features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Compute Tversky similarities.

        Parameters
        ----------
        x : (batch, in_features) or (batch, seq_len, in_features)

        Returns
        -------
        Similarity scores of shape (batch, out_features) or
        (batch, seq_len, out_features).
        '''
        original_shape = x.shape

        # Flatten sequence dimension if present.
        if x.dim() == 3:
            x = x.reshape(-1, original_shape[-1])

        # -----------------------------------------------------------------------
        # Project inputs and prototypes onto the feature bank.
        # a_dot_f[b, f] = x[b] . features[f]        -> (B, F)
        # b_dot_f[p, f] = prototypes[p] . features[f] -> (P, F)
        # Unsqueeze to (B, 1, F) and (1, P, F) so all subsequent ops
        # broadcast cleanly to (B, P, F).
        # -----------------------------------------------------------------------

        a_dot_f = (x @ self.features.t()).unsqueeze(1)             # (B, 1, F)
        b_dot_f = (self.prototypes @ self.features.t()).unsqueeze(0) # (1, P, F)

        a_has_f     = a_dot_f > 0                      # (B, 1, F)
        b_has_f     = b_dot_f > 0                      # (1, P, F)
        common_mask = a_has_f & b_has_f                # (B, P, F)

        intersection = self._compute_intersection(a_dot_f, b_dot_f)
        f_A_int_B    = (intersection * common_mask).sum(-1)  # (B, P)

        f_A_minus_B, f_B_minus_A = self._compute_differences(
            a_dot_f, b_dot_f, a_has_f, b_has_f, common_mask
        )

        similarity = (
            self.theta.abs() * f_A_int_B
            - self.alpha.abs() * f_A_minus_B
            - self.beta.abs()  * f_B_minus_A
        )  # (B, P)

        if len(original_shape) == 3:
            similarity = similarity.reshape(*original_shape[:-1], self.out_features)
        return similarity

    def _compute_intersection(
        self, a_dot_f: torch.Tensor, b_dot_f: torch.Tensor
    ) -> torch.Tensor:
        '''Compute intersection measure. Both tensors broadcast to (B, P, F).'''
        if self.intersection_reduction == 'product':
            return a_dot_f * b_dot_f
        elif self.intersection_reduction == 'min':
            return torch.min(a_dot_f, b_dot_f)
        elif self.intersection_reduction == 'max':
            return torch.max(a_dot_f, b_dot_f)
        elif self.intersection_reduction == 'mean':
            return (a_dot_f + b_dot_f) / 2
        else:
            raise ValueError(f'Unknown intersection_reduction: {self.intersection_reduction}')

    def _compute_differences(
        self,
        a_dot_f:     torch.Tensor,
        b_dot_f:     torch.Tensor,
        a_has_f:     torch.Tensor,
        b_has_f:     torch.Tensor,
        common_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        '''Compute distinctive feature measures. Returns (f_A-B, f_B-A), each (B, P).'''
        if self.difference_reduction == 'ignorematch':
            # Features exclusively present in one set.
            f_A_minus_B = (a_dot_f * ( a_has_f & ~b_has_f)).sum(-1)
            f_B_minus_A = (b_dot_f * (~a_has_f &  b_has_f)).sum(-1)
        else:  # subtractmatch
            # Strength advantage for features shared by both sets.
            f_A_minus_B = ((a_dot_f - b_dot_f) * (common_mask & (a_dot_f > b_dot_f))).sum(-1)
            f_B_minus_A = ((b_dot_f - a_dot_f) * (common_mask & (b_dot_f > a_dot_f))).sum(-1)
        return f_A_minus_B, f_B_minus_A

### 3. Experiment 1 -- XOR Classification

In [ ]:
class TverskyXORNet(nn.Module):
    '''Single Tversky layer network for XOR classification.'''
    def __init__(self, num_features: int):
        super().__init__()
        self.tversky = TverskyProjectionLayer(
            in_features=2, out_features=2, num_features=num_features
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.tversky(x)


def run_xor_experiment(
    feature_counts: List[int] = [1, 2, 4, 8, 16, 32, 64]
) -> List[Dict]:
    '''
    Test Tversky layer performance on XOR with varying feature counts.
    XOR is non-linearly separable, making it a clean test of expressive power.
    '''
    print('-' * 60)
    print('Experiment 1: XOR Classification')
    print('-' * 60)

    X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32, device=device)
    y = torch.tensor([0, 1, 1, 0], device=device)

    results = []
    for num_features in feature_counts:
        print(f'  {num_features} features...')
        model     = TverskyXORNet(num_features).to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.1)
        criterion = nn.CrossEntropyLoss()

        for _ in range(1000):
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            predictions = model(X).argmax(dim=1)
            accuracy    = (predictions == y).float().mean().item()

        results.append({'num_features': num_features, 'accuracy': accuracy,
                         'final_loss': loss.item()})
        print(f'    accuracy : {accuracy:.1%}')
        clean_memory()

    return results


def plot_xor_results(results: List[Dict]):
    '''Visualize XOR experiment results.'''
    feature_counts = [r['num_features'] for r in results]
    accuracies     = [r['accuracy']     for r in results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    ax1.plot(feature_counts, accuracies, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Features (|Omega|)')
    ax1.set_ylabel('Final Accuracy')
    ax1.set_title('XOR Classification: Accuracy vs Feature Count')
    ax1.set_xscale('log', base=2)
    ax1.set_xticks(feature_counts)
    ax1.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax1.axhline(y=1.0, color='r', linestyle='--', alpha=0.7, label='Perfect Score')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.axis('tight')
    ax2.axis('off')
    table = ax2.table(
        cellText=[[str(r['num_features']), f'{r["accuracy"]:.1%}'] for r in results],
        colLabels=['Features', 'Accuracy'],
        cellLoc='center', loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    ax2.set_title('Results Summary')

    plt.tight_layout()
    plt.show()

### 4. Experiment 2 -- Biology Q&A Dataset

In [ ]:
def load_vocabulary(
    cache_path: str,
) -> Tuple[Optional[Dict], Optional[Dict], Optional[Dict]]:
    '''Load vocabulary and token mappings from a JSON cache file.'''
    try:
        with open(cache_path, 'r') as f:
            cache_data = json.load(f)
    except FileNotFoundError:
        print(f'ERROR: Vocabulary file not found at {cache_path}')
        return None, None, None

    vocab_data = {
        'base_chars':       cache_data['base_chars'],
        'ngrams':           cache_data['ngrams'],
        'vocab_size':       cache_data['vocab_size'],
        'special_tokens':   cache_data['special_tokens'],
        'final_vocab_list': cache_data['final_vocab_list'],
    }

    # -----------------------------------------------------------------------
    # Build char_to_id: PAD=0, then enumerate vocab_list starting at 1,
    # then overwrite with explicit special token IDs from the cache.
    # -----------------------------------------------------------------------

    char_to_id = {tok: i for i, tok in enumerate(cache_data['final_vocab_list'], start=1)}
    char_to_id['<PAD>'] = 0
    for name, token_id in cache_data['special_tokens'].items():
        char_to_id[f'<{name}>'] = token_id

    id_to_char = {v: k for k, v in char_to_id.items()}
    return vocab_data, char_to_id, id_to_char


def simple_tokenize(text: str, char_to_id: Dict, ngrams: List[str]) -> List[int]:
    '''
    Greedy n-gram tokenizer. Longer n-grams are tried first; falls back to
    individual characters for any unmatched position.
    '''
    tokens        = []
    i             = 0
    sorted_ngrams = sorted(ngrams, key=len, reverse=True)
    while i < len(text):
        matched = False
        for ngram in sorted_ngrams:
            if text[i:i + len(ngram)] == ngram and ngram in char_to_id:
                tokens.append(char_to_id[ngram])
                i      += len(ngram)
                matched = True
                break
        if not matched:
            char = text[i]
            if char in char_to_id:
                tokens.append(char_to_id[char])
            i += 1
    return tokens


def create_qa_dataset(
    csv_path:   str,
    char_to_id: Dict,
    ngrams:     List[str],
    seq_len:    int = 64,
) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
    '''Create tokenized input/target pairs from biology Q&A CSV data.'''
    try:
        df = pd.read_csv(csv_path).fillna('')
    except FileNotFoundError:
        print(f'ERROR: Dataset file not found at {csv_path}')
        return None, None

    pad_id = char_to_id['<PAD>']
    special = {
        'start':    char_to_id['<START_TOKEN>'],
        'question': char_to_id['<QUESTION_TOKEN>'],
        'answer':   char_to_id['<ANSWER_TOKEN>'],
        'end':      char_to_id['<END_TOKEN>'],
    }

    inputs, targets = [], []
    for _, row in df.iterrows():
        q_tokens = simple_tokenize(row['Question'].lower(), char_to_id, ngrams)
        a_tokens = simple_tokenize(row['Answer'].lower(),   char_to_id, ngrams)
        if not q_tokens or not a_tokens:
            continue

        full_seq = (
            [special['start'], special['question']] +
            q_tokens +
            [special['answer']] +
            a_tokens +
            [special['end']]
        )
        if len(full_seq) < 2:
            continue

        inp = full_seq[:-1]
        tgt = full_seq[1:]

        # Pad or truncate to seq_len.
        inp = (inp[:seq_len] + [pad_id] * seq_len)[:seq_len]
        tgt = (tgt[:seq_len] + [pad_id] * seq_len)[:seq_len]
        inputs.append(inp)
        targets.append(tgt)

    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


class BiologyTransformer(nn.Module):
    '''Minimal transformer for biology Q&A with a Tversky output head.'''

    def __init__(self, vocab_size: int, hidden_dim: int, num_features: int):
        super().__init__()
        self.vocab_size  = vocab_size
        self.embedding   = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embed   = nn.Embedding(512, hidden_dim)
        self.transformer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            dim_feedforward=hidden_dim * 2,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.output_head = TverskyProjectionLayer(hidden_dim, vocab_size, num_features)
        self.dropout     = nn.Dropout(0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S     = x.shape
        positions = torch.arange(S, device=x.device).unsqueeze(0).expand(B, -1)
        h         = self.dropout(self.embedding(x) + self.pos_embed(positions))
        causal_mask = nn.Transformer.generate_square_subsequent_mask(S).to(x.device)
        h           = self.transformer(h, src_mask=causal_mask)
        return self.output_head(h)


def run_biology_experiment(
    vocab_data:     Dict,
    char_to_id:     Dict,
    csv_path:       str,
    feature_counts: List[int] = [2, 4, 8, 16, 32, 48, 64, 96],
    batch_size:     int = 128,
) -> List[Dict]:
    '''
    Test Tversky layer on biology Q&A with varying feature counts.
    Trains a small transformer with a Tversky output head for each count.
    '''
    print('-' * 60)
    print('Experiment 2: Biology Q&A Language Modeling')
    print('-' * 60)

    print('Loading and preprocessing data...')
    inputs, targets = create_qa_dataset(csv_path, char_to_id, vocab_data['ngrams'])
    if inputs is None:
        return []

    split_idx        = int(len(inputs) * 0.8)
    train_inputs     = inputs[:split_idx]
    train_targets    = targets[:split_idx]
    test_inputs      = inputs[split_idx:]
    test_targets     = targets[split_idx:]
    pad_id           = char_to_id['<PAD>']

    print(f'  Train : {len(train_inputs)} sequences  |  Test : {len(test_inputs)} sequences')
    print(f'  Vocab : {vocab_data["vocab_size"]}')

    results = []

    for num_features in feature_counts:
        print(f'\n  num_features = {num_features}')

        model = BiologyTransformer(
            vocab_size=vocab_data['vocab_size'],
            hidden_dim=64,
            num_features=num_features,
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

        model.train()
        for _ in tqdm(range(25), desc='  Training'):
            perm = torch.randperm(train_inputs.size(0))
            for i in range(0, len(train_inputs), batch_size):
                batch_x = train_inputs[perm[i:i + batch_size]].to(device)
                batch_y = train_targets[perm[i:i + batch_size]].to(device)
                optimizer.zero_grad()
                loss = criterion(
                    model(batch_x).view(-1, vocab_data['vocab_size']),
                    batch_y.view(-1),
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

        # -----------------------------------------------------------------------
        # Evaluation -- use reduction='sum' and track token count separately
        # so the average is over tokens, not sequences, matching training loss.
        # -----------------------------------------------------------------------

        model.eval()
        total_loss        = 0.0
        total_loss_tokens = 0
        total_correct     = 0
        total_acc_tokens  = 0

        with torch.no_grad():
            for i in range(0, len(test_inputs), batch_size):
                batch_x = test_inputs[i:i + batch_size].to(device)
                batch_y = test_targets[i:i + batch_size].to(device)
                logits  = model(batch_x)

                total_loss += F.cross_entropy(
                    logits.view(-1, vocab_data['vocab_size']),
                    batch_y.view(-1),
                    ignore_index=pad_id,
                    reduction='sum',
                ).item()

                non_pad           = batch_y != pad_id
                total_loss_tokens += non_pad.sum().item()

                predictions    = logits.argmax(dim=-1)
                total_correct += ((predictions == batch_y) & non_pad).sum().item()
                total_acc_tokens += non_pad.sum().item()

        avg_loss = total_loss       / max(total_loss_tokens, 1)
        accuracy = total_correct    / max(total_acc_tokens,  1)

        results.append({'num_features': num_features, 'accuracy': accuracy, 'test_loss': avg_loss})
        print(f'    test loss : {avg_loss:.4f}  |  accuracy : {accuracy:.2%}')
        clean_memory()

    return results


def plot_biology_results(results: List[Dict]):
    '''Visualize biology experiment results with dual plots.'''
    if not results:
        print('No results to plot.')
        return

    feature_counts = [r['num_features'] for r in results]
    accuracies     = [r['accuracy']     for r in results]
    losses         = [r['test_loss']    for r in results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    ax1.plot(feature_counts, accuracies, 'ro-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Features (|Omega|)')
    ax1.set_ylabel('Test Accuracy')
    ax1.set_title('Biology Q&A: Accuracy vs Feature Count')
    ax1.grid(True, alpha=0.3)
    for i, acc in enumerate(accuracies):
        ax1.annotate(f'{acc:.1%}', (feature_counts[i], acc),
                     textcoords='offset points', xytext=(0, 10), ha='center')

    ax2.plot(feature_counts, losses, 'go-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Features (|Omega|)')
    ax2.set_ylabel('Test Loss')
    ax2.set_title('Biology Q&A: Test Loss vs Feature Count')
    ax2.grid(True, alpha=0.3)
    for i, loss in enumerate(losses):
        ax2.annotate(f'{loss:.2f}', (feature_counts[i], loss),
                     textcoords='offset points', xytext=(0, 10), ha='center')

    plt.tight_layout()
    plt.show()

### 5. Run Experiments

In [ ]:
def main():
    '''
    Run all experiments and display results.
    The biology experiment is skipped gracefully if data files are absent.
    '''
    print('Tversky Layer Experiment')
    print('Reference: arXiv:2506.11035v1')
    print('-' * 70)

    # -----------------------------------------------------------------------
    # Experiment 1 -- XOR classification.
    # -----------------------------------------------------------------------

    xor_results = run_xor_experiment()

    print('\nXOR Results:')
    print(f'  {"Num Features":<14} {"Accuracy"}')
    print('  ' + '-' * 24)
    for r in xor_results:
        print(f'  {r["num_features"]:<14} {r["accuracy"]:.1%}')

    plot_xor_results(xor_results)

    # -----------------------------------------------------------------------
    # Experiment 2 -- Biology Q&A (skipped if data files are absent).
    # -----------------------------------------------------------------------

    vocab_cache_path = 'pipeline_v31_full/vocabulary/vocabulary_cache.json'
    csv_path         = 'blt_data/biology_qa_3000_augmented.csv'

    vocab_data, char_to_id, _ = load_vocabulary(vocab_cache_path)

    if vocab_data is not None:
        print('Vocabulary loaded.')
        bio_results = run_biology_experiment(vocab_data, char_to_id, csv_path)

        if bio_results:
            print('\nBiology Q&A Results:')
            print(f'  {"Num Features":<14} {"Accuracy":>12} {"Test Loss":>10}')
            print('  ' + '-' * 40)
            for r in bio_results:
                print(f'  {r["num_features"]:<14} {r["accuracy"]:>12.2%} {r["test_loss"]:>10.4f}')

            plot_biology_results(bio_results)

            best = max(bio_results, key=lambda r: r['accuracy'])
            print(f'\nOptimal : {best["num_features"]} features  |  best accuracy : {best["accuracy"]:.2%}')

            if len(bio_results) > 1:
                print('\nPerformance trend:')
                for i in range(1, len(bio_results)):
                    delta     = bio_results[i]['accuracy'] - bio_results[i - 1]['accuracy']
                    direction = 'up' if delta > 0 else 'down' if delta < 0 else 'flat'
                    print(f'  {bio_results[i - 1]["num_features"]:2d} -> {bio_results[i]["num_features"]:2d} features: '
                          f'{direction} {delta:+.1%}')
        else:
            print('No biology results -- check data loading.')
    else:
        print('Skipping biology experiment: data files not found.')


main()